# Module 6: Validation Agent

This notebook verifies the ValidationAgent, which runs investor relations checks to evaluate the factuality, tone, and completeness of generated answers, scoring them and optimizing phrasing for final publication.

In [ ]:
# Cell 1: Initialization
import sys
from loguru import logger

logger.remove()
logger.add(sys.stderr, level="INFO")

from extraction_agent import EmbeddingService
from predictive_analyst_agent import PredictiveAnalystAgent
from answer_agent import AnswerAgent
from validation_agent import ValidationAgent

embedding_service = EmbeddingService()
analyst_agent = PredictiveAnalystAgent(embedding_service)
answer_agent = AnswerAgent()
validation_agent = ValidationAgent()
print("ValidationAgent and prerequisite agents initialized successfully.")

In [ ]:
# Cell 2: Fetch, Answer, and Setup Validation Input
print("Analyzing and answering questions...")
results = await analyst_agent.run("Reliance Industries")
answerable = results["answerable"]
gaps = results["data_gaps"]
kpi_delta = results["kpi_delta"]

qa_pairs = await answer_agent.run_batch(answerable[:5])
print(f"Answering complete. {len(qa_pairs)} QAPairs ready for validation.")

In [ ]:
# Cell 3: Run Validation Batch
print("Running validation check batch...")
validation_output = await validation_agent.validate_batch(qa_pairs, kpi_delta, gaps)
print("Validation complete.")

In [ ]:
# Cell 4: Display Quality Summary and Gaps
print("=== VALIDATION OVERALL QUALITY SUMMARY ===")
print(f"Overall Quality Score: {validation_output['overall_quality_score']}")
print(f"Failed Validation Count: {validation_output['failed_count']}")

print("\n=== DETECTED DATA GAPS IN RUN ===")
for idx, gap in enumerate(validation_output["data_gaps"][:3]):
    print(f"- Gap {idx+1}: {gap.question}")
    print(f"  Reason: {gap.gap_reason}")

In [ ]:
# Cell 5: Display Before/After Cleaned Answers
print("=== DETAILED VALIDATION FEEDBACK per Q&A ===")
for idx, res in enumerate(validation_output["validated_pairs"]):
    print(f"\n#{idx+1} Question: {res.qa_pair.question}")
    print(f"Factual Score: {res.factual_score} | Tone Score: {res.tone_score} | Completeness: {res.completeness_score}")
    print(f"Passed Validation? {res.validation_passed}")
    
    # Compare before vs after if they differ
    if res.final_answer != res.qa_pair.answer:
        print(f"[BEFORE] Proposed Answer: {res.qa_pair.answer}")
        print(f"[AFTER] Cleaned Answer: {res.final_answer}")
    else:
        print(f"Final Answer (no changes needed): {res.final_answer}")
    if res.issues:
        print(f"Flagged Issues: {res.issues}")